In [1]:
import requests
import pandas as pd

## Scraping priced ipos data

In [2]:
# Prepare storage
all_data = []

years = [2021, 2022, 2023, 2024, 2025]
months = list(range(1, 13))

for year in years:
    for month in months:
        if year == 2025 and month > 9:
            break

        url = f"https://api.nasdaq.com/api/ipo/calendar?date={year}-{month:02d}"
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123"
        }
        response = requests.get(url, headers=headers)

        if response.status_code != 200:
            print(f"Skipping {year}-{month:02d} → Status:", response.status_code)
            continue

        data = response.json()
        if not data.get("data") or not data["data"].get("priced"):
            print(f"No IPOs found for {year}-{month:02d}")
            continue

        # Extract IPO rows
        for ipo in data["data"]["priced"]["rows"]:
            all_data.append(ipo)

# Convert to DataFrame
df = pd.DataFrame(all_data)
df.to_csv("nasdaq_ipos_api.csv", index=False)

In [11]:
df.shape

(2712, 5)

In [3]:
df = pd.read_csv('nasdaq_ipos_api.csv')
df.head()

,dealID,proposedTickerSymbol,companyName,proposedExchange,proposedSharePrice,sharesOffered,pricedDate,dollarValueOfSharesOffered,dealStatus
0,1111048-91821,NLSP,NLS Pharmaceutics Ltd.,NASDAQ Capital,4.15,"4,819,277",1/29/2021,"$20,000,000",Priced
1,1125317-93473,BCACU,"Apexigen, Inc.",NASDAQ Capital,10.00,"5,000,000",1/29/2021,"$50,000,000",Priced
2,1140946-95385,CFFVU,CF Acquisition Corp. V,NASDAQ Capital,10.00,"25,000,000",1/29/2021,"$250,000,000",Priced
3,1140915-95381,HMPT,Home Point Capital Inc.,NASDAQ Global Select,13.00,"7,250,000",1/29/2021,"$94,250,000",Priced
4,1141357-95437,BLUAU,BlueRiver Acquisition Corp.,NYSE,10.00,"25,000,000",1/29/2021,"$250,000,000",Priced


## Scraping filing ipos data

In [5]:
# Prepare storage
filings_data = []

years = [2021, 2022, 2023, 2024, 2025]
months = list(range(1, 13))

for year in years:
    for month in months:
        if year == 2025 and month > 9:  
            break

        url = f"https://api.nasdaq.com/api/ipo/calendar?date={year}-{month:02d}"
        headers = {"User-Agent": "Mozilla/5.0"}
        resp = requests.get(url, headers=headers)

        if resp.status_code != 200:
            print(f"Skipping {year}-{month:02d} → Status:", resp.status_code)
            continue

        data = resp.json()

        # Extract "filed" section
        if data.get("data") and data["data"].get("filed"):
            rows = data["data"]["filed"]["rows"]
            filings_data.extend(rows)
            print(f"✅ {len(rows)} filings found for {year}-{month:02d}")
        else:
            print(f"⚠️ No filings found for {year}-{month:02d}")

# Convert to DataFrame
df_filings = pd.DataFrame(filings_data)

# Save to CSV
df_filings.to_csv("nasdaq_ipos_filings.csv", index=False)
print("\nScraping complete. Saved to nasdaq_ipos_filings.csv")

✅ 146 filings found for 2021-01
✅ 227 filings found for 2021-02
✅ 246 filings found for 2021-03
✅ 88 filings found for 2021-04
✅ 79 filings found for 2021-05
✅ 107 filings found for 2021-06
✅ 103 filings found for 2021-07
✅ 59 filings found for 2021-08
✅ 82 filings found for 2021-09
✅ 114 filings found for 2021-10
✅ 86 filings found for 2021-11
✅ 54 filings found for 2021-12
✅ 48 filings found for 2022-01
✅ 21 filings found for 2022-02
✅ 34 filings found for 2022-03
✅ 32 filings found for 2022-04
✅ 18 filings found for 2022-05
✅ 21 filings found for 2022-06
✅ 18 filings found for 2022-07
✅ 22 filings found for 2022-08
✅ 30 filings found for 2022-09
✅ 16 filings found for 2022-10
✅ 21 filings found for 2022-11
✅ 21 filings found for 2022-12
✅ 22 filings found for 2023-01
✅ 29 filings found for 2023-02
✅ 32 filings found for 2023-03
✅ 15 filings found for 2023-04
✅ 14 filings found for 2023-05
✅ 24 filings found for 2023-06
✅ 21 filings found for 2023-07
✅ 26 filings found for 2023-08
✅ 

In [4]:
df.shape

(1896, 9)

In [6]:
df = pd.read_csv('nasdaq_ipos_filings.csv')
df.head()

,dealID,proposedTickerSymbol,companyName,filedDate,dollarValueOfSharesOffered
0,1143899-95750,ASPCU,Alpha Capital Acquisition Co,1/29/2021,"$200,000,000"
1,1143898-95749,TWNIU,Tailwind International Acquisition Corp.,1/29/2021,"$300,000,000"
2,1143897-95748,CFVIU,Rumble Inc.,1/29/2021,"$300,000,000"
3,1143896-95745,LGV'U,Longview Acquisition Corp. II,1/29/2021,"$600,000,000"
4,1143889-95740,SPGSU,"Simon Property Group Acquisition Holdings, Inc.",1/29/2021,"$300,000,000"


In [7]:
df.shape

(2712, 5)

## Merging the priced and filings ipo data 

In [14]:
# Load CSVs
priced_df = pd.read_csv("nasdaq_ipos_api.csv")
filings_df = pd.read_csv("nasdaq_ipos_filings.csv")

# Merge on dealID with outer join
merged_df = pd.merge(priced_df, filings_df, on="dealID", how="outer")

# Save to CSV
merged_df.to_csv("nasdaq_ipos_merged_outerjoin.csv", index=False)

print("Merged file saved as nasdaq_ipos_merged.csv")



Merged file saved as nasdaq_ipos_merged.csv


In [15]:
print("Shape of merged file:", merged_df.shape)


Shape of merged file: (2845, 13)


In [16]:
merged_df.head()

,dealID,proposedTickerSymbol_x,companyName_x,proposedExchange,proposedSharePrice,sharesOffered,pricedDate,dollarValueOfSharesOffered_x,dealStatus,proposedTickerSymbol_y,companyName_y,filedDate,dollarValueOfSharesOffered_y
0,1006008-95511,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,LumiraDx Ltd,1/15/2021,"$100,000,000"
1,1007622-101060,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.11 ABR CORP.,11/08/2021,"$100,000,000"
2,1008124-100682,TRDA,"Entrada Therapeutics, Inc.",NASDAQ Global,20.00,"9,075,000",10/29/2021,"$181,500,000",Priced,TRDA,"Entrada Therapeutics, Inc.",10/08/2021,"$181,500,000"
3,1010055-100948,NU,Nu Holdings Ltd.,NYSE,9.00,"289,150,555",12/09/2021,"$2,602,354,995",Priced,NU,Nu Holdings Ltd.,11/01/2021,"$2,602,354,995"
4,1010448-115077,LBRX,LB PHARMACEUTICALS INC,NASDAQ Global,15.00,"19,000,000",9/11/2025,"$285,000,000",Priced,LBRX,LB PHARMACEUTICALS INC,8/22/2025,"$285,000,000"
